In [11]:
import pandas as pd
import ast
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import classification_report

In [12]:
# FEATURES
features = [
    "duration",
    "login_attempts",
    "num_commands",
    "file_download",
    "recon",
    "download",
    "chmod",
    "execution",
    "persistence",
    "commands_per_second",
    "unique_commands",
    "avg_command_length",
    "auth_severity_score"
]

# LOAD
dataset = pd.read_csv("../log/dataset.csv")

# FIX labels 
dataset["attack_type"] = dataset["attack_type"].apply(ast.literal_eval)

# FEATURES
X = dataset[features].fillna(0)

# MULTILABEL ENCODING
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(dataset["attack_type"])

# SPLIT 
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# MODEL
base_model = RandomForestClassifier(
    n_estimators=150,
    random_state=42
)

model = MultiOutputClassifier(base_model)
model.fit(X_train, y_train)

# PREDICT
y_pred = model.predict(X_test)

# REPORT
print(classification_report(
    y_test,
    y_pred,
    target_names=mlb.classes_,
    zero_division=0
))

                        precision    recall  f1-score   support

auth_success_no_action       1.00      0.99      0.99        96
             execution       1.00      1.00      1.00         1
         file_activity       1.00      0.50      0.67         2
    high_auth_pressure       0.98      1.00      0.99        55
    interactive_attack       0.00      0.00      0.00         1
      malware_download       0.00      0.00      0.00         0
           persistence       1.00      1.00      1.00         1
      privilege_change       1.00      1.00      1.00         1
        reconnaissance       1.00      1.00      1.00         4
         scan_or_noise       0.99      1.00      1.00       154

             micro avg       0.99      0.99      0.99       315
             macro avg       0.80      0.75      0.76       315
          weighted avg       0.99      0.99      0.99       315
           samples avg       0.99      0.99      0.99       315

